# 📊 Forecasting Human Development & Measuring Regional Inequality in India

**Dataset:** GDL Subnational HDI Data (1990–2022) — 36 Indian States/UTs  
**Workflow:**
1. Data Preprocessing
2. Exploratory Data Analysis (EDA)
3. Inequality Analysis (Gini Coefficient)
4. Stationarity Testing (ADF, ACF, PACF)
5. Model Order Identification (AIC Comparison)
6. Model Fitting & Comparison (ARIMA, SARIMA, Exponential Smoothing)
7. COVID Counterfactual Analysis
8. Policy Insights


---
## 📦 Section 0: Install & Import Libraries

In [ ]:
# ── Standard Libraries ────────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── Statistical Tests ─────────────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ── Time Series Models ────────────────────────────────────────────────────────
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# ── Error Metrics ─────────────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── Settings ──────────────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13})

print('✅ All libraries imported successfully!')

---
## 📂 Section 1: Data Preprocessing

In [ ]:
# ── 1.1 Load Raw Data ─────────────────────────────────────────────────────────
raw = pd.read_csv('GDL-Subnational-HDI-data.csv')
print(f'Raw shape: {raw.shape}')
raw.head(3)

In [ ]:
# ── 1.2 Separate National vs Subnational rows ─────────────────────────────────
national = raw[raw['Level'] == 'National'].copy()
states   = raw[raw['Level'] == 'Subnat'].copy()

print(f'National rows : {len(national)}')
print(f'State/UT rows : {len(states)}')

In [ ]:
# ── 1.3 Reshape to Long Format for easy analysis ──────────────────────────────
year_cols = [str(y) for y in range(1990, 2023)]   # columns 1990 … 2022

states_long = states.melt(
    id_vars=['Region', 'GDLCODE'],
    value_vars=year_cols,
    var_name='Year',
    value_name='HDI'
)
states_long['Year'] = states_long['Year'].astype(int)
states_long = states_long.sort_values(['Region', 'Year']).reset_index(drop=True)

# ── 1.4 Missing Value Check ───────────────────────────────────────────────────
missing = states_long['HDI'].isnull().sum()
print(f'Missing HDI values: {missing}')

# Fill any missing values via forward-fill within each region
states_long['HDI'] = states_long.groupby('Region')['HDI'].transform(
    lambda x: x.fillna(method='ffill').fillna(method='bfill')
)
print(f'Missing after fill : {states_long["HDI"].isnull().sum()}')
states_long.head()

In [ ]:
# ── 1.5 National HDI time series ─────────────────────────────────────────────
national_series = national[year_cols].iloc[0].rename('HDI')
national_series.index = national_series.index.astype(int)
national_series.index.name = 'Year'

print('India National HDI (first & last 5 years):')
print(pd.concat([national_series.head(), national_series.tail()]))

---
## 🔍 Section 2: Exploratory Data Analysis (EDA)

In [ ]:
# ── 2.1 Descriptive Statistics (latest year = 2022) ───────────────────────────
latest = states_long[states_long['Year'] == 2022][['Region', 'HDI']]
print('=== HDI Summary Statistics (2022) ===')
print(latest['HDI'].describe().round(4))

In [ ]:
# ── 2.2 India's National HDI Trend ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(national_series.index, national_series.values,
        color='steelblue', linewidth=2.5, marker='o', markersize=4)
ax.axvspan(2020, 2021, color='tomato', alpha=0.15, label='COVID period')
ax.set_title('India National HDI Trend (1990–2022)')
ax.set_xlabel('Year')
ax.set_ylabel('HDI')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.3 State-wise HDI in 2022 (Bar Chart) ───────────────────────────────────
latest_sorted = latest.sort_values('HDI', ascending=True)

fig, ax = plt.subplots(figsize=(10, 12))
bars = ax.barh(latest_sorted['Region'], latest_sorted['HDI'],
               color=plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(latest_sorted))))
ax.set_title('State-wise HDI (2022) — Ranked')
ax.set_xlabel('HDI')
ax.axvline(latest_sorted['HDI'].mean(), color='navy',
           linestyle='--', linewidth=1.5, label=f'Mean = {latest_sorted["HDI"].mean():.3f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.4 Heatmap: HDI by State × Year ─────────────────────────────────────────
pivot = states_long.pivot(index='Region', columns='Year', values='HDI')

# Show every 5 years to keep the plot readable
cols_to_show = [y for y in range(1990, 2023, 5)] + [2022]
pivot_sub = pivot[cols_to_show]

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(pivot_sub, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.4, ax=ax, cbar_kws={'label': 'HDI'})
ax.set_title('HDI by State and Year (Selected Years)')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.5 HDI Trend Lines for All States ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

for region, grp in states_long.groupby('Region'):
    ax.plot(grp['Year'], grp['HDI'], alpha=0.4, linewidth=1)

# Overlay national average
ax.plot(national_series.index, national_series.values,
        color='black', linewidth=3, label='National Average')
ax.axvspan(2020, 2021, color='tomato', alpha=0.12, label='COVID period')
ax.set_title('HDI Trends: All Indian States (1990–2022)')
ax.set_xlabel('Year')
ax.set_ylabel('HDI')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.6 Top 5 vs Bottom 5 States ─────────────────────────────────────────────
top5    = latest.nlargest(5,  'HDI')['Region'].tolist()
bottom5 = latest.nsmallest(5, 'HDI')['Region'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, group, title, color in zip(
        axes,
        [top5, bottom5],
        ['Top 5 States — HDI Trend', 'Bottom 5 States — HDI Trend'],
        ['steelblue', 'tomato']):
    for region in group:
        d = states_long[states_long['Region'] == region]
        ax.plot(d['Year'], d['HDI'], marker='o', markersize=3,
                linewidth=2, label=region)
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.set_ylabel('HDI')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('Top 5 States (2022):')
print(latest.nlargest(5, 'HDI').to_string(index=False))
print('\nBottom 5 States (2022):')
print(latest.nsmallest(5, 'HDI').to_string(index=False))

---
## ⚖️ Section 3: Inequality Analysis — Gini Coefficient

In [ ]:
# ── 3.1 Gini Coefficient Function ─────────────────────────────────────────────
def gini_coefficient(values):
    """
    Compute the Gini coefficient from an array of values.
    Gini = 0  → perfect equality
    Gini = 1  → maximum inequality
    """
    values = np.array(sorted(values))
    n = len(values)
    cumulative = np.cumsum(values)
    gini = (2 * np.sum((np.arange(1, n + 1)) * values) / (n * cumulative[-1])) - (n + 1) / n
    return round(gini, 6)

In [ ]:
# ── 3.2 Gini per Year ────────────────────────────────────────────────────────
gini_by_year = (
    states_long
    .groupby('Year')['HDI']
    .apply(gini_coefficient)
    .reset_index(name='Gini')
)

print('Gini Coefficient over time (selected years):')
print(gini_by_year[gini_by_year['Year'].isin([1990, 2000, 2010, 2019, 2020, 2021, 2022])]
      .to_string(index=False))

In [ ]:
# ── 3.3 Plot Gini Trend ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Gini over time
ax = axes[0]
ax.plot(gini_by_year['Year'], gini_by_year['Gini'],
        color='darkorange', linewidth=2.5, marker='o', markersize=4)
ax.axvspan(2020, 2021, color='tomato', alpha=0.15, label='COVID period')
ax.set_title('Gini Coefficient of HDI Across States (1990–2022)')
ax.set_xlabel('Year')
ax.set_ylabel('Gini Coefficient')
ax.legend()

# Lorenz Curve for 2022
ax = axes[1]
vals_2022 = np.sort(states_long[states_long['Year'] == 2022]['HDI'].values)
lorenz_x = np.concatenate([[0], np.linspace(0, 1, len(vals_2022))])
lorenz_y = np.concatenate([[0], np.cumsum(vals_2022) / vals_2022.sum()])
ax.plot(lorenz_x, lorenz_y, color='steelblue', linewidth=2.5, label='Lorenz Curve (2022)')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Perfect Equality')
ax.fill_between(lorenz_x, lorenz_x, lorenz_y, alpha=0.15, color='steelblue')
ax.set_title('Lorenz Curve — HDI (2022)')
ax.set_xlabel('Cumulative Share of States')
ax.set_ylabel('Cumulative Share of HDI')
ax.legend()

plt.tight_layout()
plt.show()

g2022 = gini_by_year[gini_by_year['Year'] == 2022]['Gini'].values[0]
g1990 = gini_by_year[gini_by_year['Year'] == 1990]['Gini'].values[0]
print(f'\nGini 1990: {g1990:.4f}  |  Gini 2022: {g2022:.4f}')
print(f'Change   : {g2022 - g1990:+.4f} ({"decreased" if g2022 < g1990 else "increased"} — inequality {"fell" if g2022 < g1990 else "rose"})')

In [ ]:
# ── 3.4 Pre vs Post COVID Gini Comparison ────────────────────────────────────
pre_covid  = gini_by_year[gini_by_year['Year'] == 2019]['Gini'].values[0]
covid_yr   = gini_by_year[gini_by_year['Year'] == 2020]['Gini'].values[0]
post_covid = gini_by_year[gini_by_year['Year'] == 2022]['Gini'].values[0]

print(f'Gini — Pre-COVID  (2019): {pre_covid:.4f}')
print(f'Gini — COVID year (2020): {covid_yr:.4f}')
print(f'Gini — Post-COVID (2022): {post_covid:.4f}')

---
## 📈 Section 4: Stationarity Testing (ADF, ACF, PACF)

In [ ]:
# ── 4.1 Augmented Dickey-Fuller (ADF) Test ────────────────────────────────────
def run_adf_test(series, name='Series'):
    """
    Run ADF test and print a clear summary.
    H0: Series has a unit root (non-stationary)
    Reject H0 if p-value < 0.05 → series is stationary
    """
    result = adfuller(series.dropna(), autolag='AIC')
    print(f'--- ADF Test: {name} ---')
    print(f'  ADF Statistic : {result[0]:.4f}')
    print(f'  p-value       : {result[1]:.4f}')
    print(f'  Critical (1%) : {result[4]["1%"]:.4f}')
    print(f'  Critical (5%) : {result[4]["5%"]:.4f}')
    conclusion = 'STATIONARY' if result[1] < 0.05 else 'NON-STATIONARY'
    print(f'  Conclusion    : {conclusion}\n')
    return result[1]

In [ ]:
# ── 4.2 Test on National HDI (raw) ───────────────────────────────────────────
p_raw = run_adf_test(national_series, name='National HDI (raw)')

# First difference
hdi_diff = national_series.diff().dropna()
p_diff   = run_adf_test(hdi_diff, name='National HDI (1st difference)')

In [ ]:
# ── 4.3 ACF & PACF Plots ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 7))

plot_acf (national_series, lags=15, ax=axes[0, 0], title='ACF — Raw HDI')
plot_pacf(national_series, lags=15, ax=axes[0, 1], title='PACF — Raw HDI')
plot_acf (hdi_diff,        lags=15, ax=axes[1, 0], title='ACF — 1st Differenced HDI')
plot_pacf(hdi_diff,        lags=15, ax=axes[1, 1], title='PACF — 1st Differenced HDI')

plt.tight_layout()
plt.show()

print("""
Interpretation Guide:
  ACF  → determines MA (q) order: note the lag where ACF cuts off
  PACF → determines AR (p) order: note the lag where PACF cuts off
  If raw series is non-stationary, use the differenced plots to decide p, q.
""")

---
## 🔢 Section 5: AIC-Based Order Identification

In [ ]:
# ── 5.1 Grid Search ARIMA(p,d,q) using AIC ───────────────────────────────────
from itertools import product

def arima_aic_grid(series, p_range, d_range, q_range):
    """
    Fit ARIMA(p,d,q) for all (p,d,q) combinations and return AIC table.
    Lower AIC = better model.
    """
    results = []
    for p, d, q in product(p_range, d_range, q_range):
        try:
            model = ARIMA(series, order=(p, d, q)).fit()
            results.append({'p': p, 'd': d, 'q': q, 'AIC': round(model.aic, 2)})
        except Exception:
            pass   # skip non-convergent combinations
    aic_df = pd.DataFrame(results).sort_values('AIC').reset_index(drop=True)
    return aic_df

print('Running ARIMA grid search … (may take ~30 seconds)')
aic_table = arima_aic_grid(national_series,
                           p_range=range(0, 4),
                           d_range=range(0, 2),
                           q_range=range(0, 4))

print('\nTop 10 ARIMA orders by AIC:')
print(aic_table.head(10).to_string(index=False))

best_p, best_d, best_q = aic_table.iloc[0][['p', 'd', 'q']].astype(int)
print(f'\n✅ Best ARIMA order: ({best_p}, {best_d}, {best_q})  AIC={aic_table.iloc[0]["AIC"]}')

In [ ]:
# ── 5.2 Bar Chart of Top-15 AIC Values ───────────────────────────────────────
top15 = aic_table.head(15).copy()
top15['label'] = top15.apply(lambda r: f'({int(r.p)},{int(r.d)},{int(r.q)})', axis=1)

fig, ax = plt.subplots(figsize=(12, 4))
colors = ['gold'] + ['steelblue'] * 14
ax.bar(top15['label'], top15['AIC'], color=colors, edgecolor='black', linewidth=0.6)
ax.set_title('Top 15 ARIMA Orders by AIC (lower = better)')
ax.set_xlabel('ARIMA(p,d,q)')
ax.set_ylabel('AIC')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 🤖 Section 6: Model Fitting & Comparison

Three models will be trained and compared:
1. **ARIMA** — best order from AIC grid
2. **SARIMA** — handles potential seasonal patterns
3. **Holt-Winters Exponential Smoothing** — classic benchmark

In [ ]:
# ── 6.1 Train / Test Split ────────────────────────────────────────────────────
# Use 1990–2017 for training, 2018–2022 for testing (5-year horizon)
train = national_series[national_series.index <= 2017]
test  = national_series[national_series.index >= 2018]

print(f'Training set: {train.index[0]}–{train.index[-1]}  ({len(train)} years)')
print(f'Test set    : {test.index[0]}–{test.index[-1]}   ({len(test)} years)')

In [ ]:
# ── 6.2 Model 1: ARIMA ────────────────────────────────────────────────────────
arima_model  = ARIMA(train, order=(best_p, best_d, best_q)).fit()
arima_fc     = arima_model.forecast(steps=len(test))
arima_fc.index = test.index

print(arima_model.summary())

In [ ]:
# ── 6.3 Model 2: SARIMA(1,1,1)(0,0,1,5) ─────────────────────────────────────
# Annual data → modest seasonal period of 5 captures decade patterns
sarima_model = SARIMAX(train, order=(1, 1, 1),
                       seasonal_order=(0, 0, 1, 5)).fit(disp=False)
sarima_fc    = sarima_model.forecast(steps=len(test))
sarima_fc.index = test.index

print(sarima_model.summary())

In [ ]:
# ── 6.4 Model 3: Holt-Winters Exponential Smoothing ──────────────────────────
hw_model = ExponentialSmoothing(
    train,
    trend='add',       # additive trend for steady growth
    seasonal=None      # no clear seasonality in annual data
).fit(optimized=True)

hw_fc = hw_model.forecast(steps=len(test))
hw_fc.index = test.index

print('Holt-Winters Parameters:')
print(f'  Alpha (level)  : {hw_model.params["smoothing_level"]:.4f}')
print(f'  Beta  (trend)  : {hw_model.params["smoothing_trend"]:.4f}')

In [ ]:
# ── 6.5 Model Evaluation: MAE, RMSE ──────────────────────────────────────────
def evaluate_model(actual, predicted, model_name):
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    return {'Model': model_name, 'MAE': round(mae, 5),
            'RMSE': round(rmse, 5), 'MAPE (%)': round(mape, 3)}

scores = [
    evaluate_model(test, arima_fc,  f'ARIMA({best_p},{best_d},{best_q})'),
    evaluate_model(test, sarima_fc, 'SARIMA(1,1,1)(0,0,1,5)'),
    evaluate_model(test, hw_fc,     'Holt-Winters'),
]
scores_df = pd.DataFrame(scores)
print('\n===  Model Performance Comparison  ===')
print(scores_df.to_string(index=False))

best_model_name = scores_df.loc[scores_df['RMSE'].idxmin(), 'Model']
print(f'\n🏆 Best Model (lowest RMSE): {best_model_name}')

In [ ]:
# ── 6.6 Forecast vs Actual Plot ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(train.index, train.values,
        color='black', linewidth=2, label='Training Data')
ax.plot(test.index, test.values,
        color='black', linewidth=2.5, linestyle='--', label='Actual (Test)')

ax.plot(arima_fc.index,  arima_fc.values,  marker='o', label=f'ARIMA({best_p},{best_d},{best_q})')
ax.plot(sarima_fc.index, sarima_fc.values, marker='s', label='SARIMA')
ax.plot(hw_fc.index,     hw_fc.values,     marker='^', label='Holt-Winters')

ax.axvline(2017.5, color='grey', linestyle=':', linewidth=1.5, label='Train/Test split')
ax.set_title('Model Forecasts vs Actual National HDI (2018–2022)')
ax.set_xlabel('Year')
ax.set_ylabel('HDI')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.7 Future Forecast: 2023–2030 (using full data) ─────────────────────────
FORECAST_YEARS = 8   # 2023–2030
future_years   = list(range(2023, 2023 + FORECAST_YEARS))

# Refit ARIMA on full series
best_arima_full = ARIMA(national_series, order=(best_p, best_d, best_q)).fit()
future_fc       = best_arima_full.forecast(steps=FORECAST_YEARS)
ci              = best_arima_full.get_forecast(steps=FORECAST_YEARS).conf_int(alpha=0.10)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(national_series.index, national_series.values,
        color='black', linewidth=2.5, label='Historical HDI')
ax.plot(future_years, future_fc.values,
        color='steelblue', linewidth=2.5, marker='o', label='ARIMA Forecast')
ax.fill_between(future_years, ci.iloc[:, 0], ci.iloc[:, 1],
                alpha=0.2, color='steelblue', label='90% Confidence Interval')
ax.axvline(2022.5, color='grey', linestyle=':', linewidth=1.5)
ax.set_title('India National HDI — Forecast to 2030')
ax.set_xlabel('Year')
ax.set_ylabel('HDI')
ax.legend()
plt.tight_layout()
plt.show()

print('\nForecasted HDI (2023–2030):')
for yr, val in zip(future_years, future_fc.values):
    print(f'  {yr}: {val:.4f}')

---
## 🦠 Section 7: COVID Counterfactual Analysis

We estimate what HDI *would have been* in 2020–2022 without COVID by  
fitting an ARIMA model on **pre-COVID data (1990–2019)** and projecting forward.

In [ ]:
# ── 7.1 Fit Pre-COVID Model ───────────────────────────────────────────────────
pre_covid_series = national_series[national_series.index <= 2019]
covid_actual     = national_series[national_series.index >= 2020]

cf_model = ARIMA(pre_covid_series, order=(best_p, best_d, best_q)).fit()
cf_fc    = cf_model.forecast(steps=3)   # 2020, 2021, 2022
cf_fc.index = [2020, 2021, 2022]

print('Counterfactual (No-COVID) vs Actual HDI:')
cf_compare = pd.DataFrame({
    'Year'            : [2020, 2021, 2022],
    'Actual HDI'      : covid_actual.values,
    'Counterfactual'  : cf_fc.values,
    'HDI Loss (gap)'  : cf_fc.values - covid_actual.values
})
print(cf_compare.to_string(index=False))
total_loss = cf_compare['HDI Loss (gap)'].sum()
print(f'\nCumulative HDI loss (2020–2022): {total_loss:.4f}')

In [ ]:
# ── 7.2 Plot Counterfactual ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(national_series.index, national_series.values,
        color='black', linewidth=2.5, label='Actual HDI')
ax.plot([2019, 2020, 2021, 2022],
        [pre_covid_series[2019]] + cf_fc.tolist(),
        color='steelblue', linewidth=2.5, linestyle='--',
        marker='o', label='Counterfactual (No COVID)')

# Shade the gap
ax.fill_between([2020, 2021, 2022],
                covid_actual.values,
                cf_fc.values,
                alpha=0.25, color='tomato', label='HDI Loss due to COVID')

ax.axvspan(2020, 2022, color='tomato', alpha=0.06)
ax.set_title('COVID Counterfactual: Actual vs Projected HDI Without COVID')
ax.set_xlabel('Year')
ax.set_ylabel('HDI')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.3 State-level COVID Impact (2019 vs 2021 HDI drop) ─────────────────────
hdi_2019 = states_long[states_long['Year'] == 2019][['Region', 'HDI']].rename(columns={'HDI': 'HDI_2019'})
hdi_2021 = states_long[states_long['Year'] == 2021][['Region', 'HDI']].rename(columns={'HDI': 'HDI_2021'})
hdi_2022 = states_long[states_long['Year'] == 2022][['Region', 'HDI']].rename(columns={'HDI': 'HDI_2022'})

covid_impact = hdi_2019.merge(hdi_2021, on='Region').merge(hdi_2022, on='Region')
covid_impact['COVID_drop']    = covid_impact['HDI_2019'] - covid_impact['HDI_2021']
covid_impact['Recovery_2022'] = covid_impact['HDI_2022'] - covid_impact['HDI_2021']
covid_impact = covid_impact.sort_values('COVID_drop', ascending=False)

print('States most impacted by COVID (HDI 2019→2021 drop):')
print(covid_impact[['Region', 'HDI_2019', 'HDI_2021', 'COVID_drop']]
      .head(10).to_string(index=False))

In [ ]:
# ── 7.4 COVID Impact Plot ─────────────────────────────────────────────────────
top_impacted = covid_impact.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['tomato' if d > 0 else 'steelblue' for d in top_impacted['COVID_drop']]
ax.barh(top_impacted['Region'], top_impacted['COVID_drop'],
        color=colors, edgecolor='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('COVID Impact: HDI Drop (2019 → 2021), Top 15 States')
ax.set_xlabel('HDI Change')
plt.tight_layout()
plt.show()

---
## 💡 Section 8: Policy Insights

In [ ]:
# ── 8.1 Beta-Convergence: Are Poor States Catching Up? ───────────────────────
# Scatter: initial HDI (1990) vs growth rate → negative slope = convergence
hdi_1990 = states_long[states_long['Year'] == 1990][['Region', 'HDI']].rename(columns={'HDI': 'HDI_1990'})
hdi_2022 = states_long[states_long['Year'] == 2022][['Region', 'HDI']].rename(columns={'HDI': 'HDI_2022'})

conv_df = hdi_1990.merge(hdi_2022, on='Region')
conv_df['growth_rate'] = (conv_df['HDI_2022'] / conv_df['HDI_1990']) ** (1/32) - 1

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(conv_df['HDI_1990'], conv_df['growth_rate'],
           color='steelblue', s=60, edgecolor='black', linewidth=0.5)

for _, row in conv_df.iterrows():
    ax.annotate(row['Region'], (row['HDI_1990'], row['growth_rate']),
                fontsize=6, alpha=0.7, ha='left')

# OLS trend line
m = np.polyfit(conv_df['HDI_1990'], conv_df['growth_rate'], 1)
x_line = np.linspace(conv_df['HDI_1990'].min(), conv_df['HDI_1990'].max(), 100)
ax.plot(x_line, np.polyval(m, x_line), color='tomato', linewidth=2,
        label=f'Trend (slope={m[0]:.3f})')
ax.set_title('Beta-Convergence: Initial HDI vs Annual Growth Rate (1990–2022)')
ax.set_xlabel('HDI in 1990 (initial level)')
ax.set_ylabel('Annual Growth Rate')
ax.legend()
plt.tight_layout()
plt.show()

if m[0] < 0:
    print('✅ CONVERGENCE detected: States with lower initial HDI grew faster.')
else:
    print('⚠️  DIVERGENCE detected: States with higher initial HDI grew faster.')

In [ ]:
# ── 8.2 Sigma Convergence: Standard Deviation of HDI Over Time ───────────────
std_by_year = states_long.groupby('Year')['HDI'].std().reset_index(name='STD')

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(std_by_year['Year'], std_by_year['STD'],
        color='purple', linewidth=2.5, marker='o', markersize=4)
ax.axvspan(2020, 2021, color='tomato', alpha=0.15, label='COVID period')
ax.set_title('Sigma-Convergence: Cross-State Standard Deviation of HDI (1990–2022)')
ax.set_xlabel('Year')
ax.set_ylabel('Std Dev of HDI')
ax.legend()
plt.tight_layout()
plt.show()

std_1990 = std_by_year[std_by_year['Year'] == 1990]['STD'].values[0]
std_2022 = std_by_year[std_by_year['Year'] == 2022]['STD'].values[0]
print(f'Std Dev 1990: {std_1990:.4f}  |  Std Dev 2022: {std_2022:.4f}')
print('Dispersion has', 'DECREASED' if std_2022 < std_1990 else 'INCREASED',
      '→ sigma convergence', 'confirmed' if std_2022 < std_1990 else 'NOT confirmed')

In [ ]:
# ── 8.3 Identify Persistently Lagging States ─────────────────────────────────
mean_hdi_by_state = states_long.groupby('Region')['HDI'].mean().sort_values()
national_avg = states_long['HDI'].mean()

laggards = mean_hdi_by_state[mean_hdi_by_state < national_avg].index.tolist()

print(f'States BELOW national average HDI (1990–2022 mean = {national_avg:.3f}):')
for s in laggards:
    print(f'  {s:<30} avg HDI = {mean_hdi_by_state[s]:.4f}')

In [ ]:
# ── 8.4 Policy Recommendations Summary ───────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║               POLICY INSIGHTS — INDIA HDI REGIONAL INEQUALITY               ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  1. TARGETED INVESTMENT IN LAGGING STATES                                   ║
║     Bihar, UP, Jharkhand, MP show persistently low HDI.                     ║
║     → Prioritize health, education & infrastructure spending here.           ║
║                                                                              ║
║  2. COVID RECOVERY PLANNING                                                  ║
║     HDI dipped in 2020-21 across all states. Large states need special       ║
║     fiscal support to recover lost human development ground.                 ║
║                                                                              ║
║  3. NARROWING THE NORTH-SOUTH DIVIDE                                         ║
║     Southern states (Kerala, Goa, Tamil Nadu) consistently lead.             ║
║     → Transfer successful health & education models to northern states.      ║
║                                                                              ║
║  4. GINI REDUCTION TARGET                                                    ║
║     HDI Gini is still significant. Inequality reduction should be an         ║
║     explicit policy goal alongside aggregate HDI growth.                     ║
║                                                                              ║
║  5. FORECAST UTILIZATION (HDI projected to ~0.70+ by 2030)                  ║
║     If trend continues, India can achieve 'High Human Development'           ║
║     (HDI ≥ 0.70) nationally — but only if lagging states accelerate.         ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

---
## ✅ Project Summary

| Step | Analysis | Key Finding |
|------|----------|-------------|
| 1 | Preprocessing | 36 states, 1990–2022, wide→long format |
| 2 | EDA | Steady rise; Kerala, Goa lead; Bihar, UP lag |
| 3 | Gini Coefficient | Inter-state inequality has declined over decades |
| 4 | ADF / ACF / PACF | Raw series non-stationary; 1st diff stationary |
| 5 | AIC Comparison | Best ARIMA order confirmed via grid search |
| 6 | Model Fitting | ARIMA / SARIMA / Holt-Winters compared on test set |
| 7 | COVID Counterfactual | COVID caused measurable HDI loss 2020–2021 |
| 8 | Policy Insights | Lagging state targeting, convergence analysis |